In [1]:
direct = '/glade/campaign/collections/rda/data/d633000/e5.oper.invariant/197901/e5.oper.invariant.128_026_cl.ll025sc.1979010100_1979010100.nc '



In [37]:
from dask_jobqueue import SLURMCluster
import glob
import os
import xarray as xr 
import pandas as pd
import numpy as np
from dask.distributed import Client


In [38]:
cluster = SLURMCluster(
    job_name="Climt1",          # --job-name
    cores=42,                   # 24 cores per node
    processes=6,                # One process per task
    memory="64GB",             # --mem
    walltime="02:00:00",        # --time
    queue="med",                # --partition
    log_directory=".",          # Logs will be saved to the current directory
)

from dask.distributed import Client


cluster.scale(4) # Adjust the number of workers dynamically
client = Client(cluster)
client

/home1/nalex2023/.local/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36261 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.42.239.61:36261/status,
Dashboard: http://10.42.239.61:36261/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.42.239.61:41653,Workers: 0
Dashboard: http://10.42.239.61:36261/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [ ]:
cluster.close()

: 

In [ ]:
client.shutdown()

In [3]:
"""
from dask_jobqueue import PBSCluster
    
cluster = PBSCluster(
    job_name = 'ml_test_trop',
    cores = 1,
    memory = '4GiB',
    processes = 1,
    local_directory = '/local_scratch/pbs.$PBS_JOBID/dask/spill',
    resource_spec = 'select=1:ncpus=1:mem=4GB',
    queue = 'casper',
    walltime = '30:00',
    interface = 'ext'
)
"""

"\nfrom dask_jobqueue import PBSCluster\n    \ncluster = PBSCluster(\n    job_name = 'ml_test_trop',\n    cores = 1,\n    memory = '4GiB',\n    processes = 1,\n    local_directory = '/local_scratch/pbs.$PBS_JOBID/dask/spill',\n    resource_spec = 'select=1:ncpus=1:mem=4GB',\n    queue = 'casper',\n    walltime = '30:00',\n    interface = 'ext'\n)\n"

In [27]:
client.shutdown()

In [10]:
print(cluster.job_script())

#!/usr/bin/env bash

#SBATCH -J Climt1
#SBATCH -e ./Climt1-%J.err
#SBATCH -o ./Climt1-%J.out
#SBATCH -p med
#SBATCH -n 1
#SBATCH --cpus-per-task=6
#SBATCH --mem=30G
#SBATCH -t 01:15:00

/usr/bin/python3 -m distributed.cli.dask_worker tcp://10.42.239.61:37375 --name dummy-name --nthreads 6 --memory-limit 29.80GiB --nanny --death-timeout 60



In [25]:
client = Client(cluster)

In [11]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.42.239.61:8787/status,
Dashboard: http://10.42.239.61:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.42.239.61:40929,Workers: 0
Dashboard: http://10.42.239.61:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [39]:
# calcualte q 5 year 
'/home2/nalex2023/Datasets/ERA_global/era5_pl_v_2005_12_13.nc'

all_dsets = glob.glob('/home2/nalex2023/Datasets/ERA_global/era5_pl_q_*.nc')

read_dft = pd.DataFrame(all_dsets,columns=['files'])

read_dft['year'] = read_dft['files'].str.split(os.sep).str[-1].str.split('_').str[3]
read_dft['month'] = read_dft['files'].str.split(os.sep).str[-1].str.split('_').str[4]
read_dft['day'] = read_dft['files'].str.split(os.sep).str[-1].str.split('_').str[5].str[0:2]

read_dft['datetime'] = pd.to_datetime(read_dft[['year','month','day']],format='%Y%m%D')

read_dft_sorted = read_dft.sort_values('datetime').reset_index(drop=True)

read_dft_sorted

,files,year,month,day,datetime
0,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2000,01,01,2000-01-01
1,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2000,01,02,2000-01-02
2,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2000,01,03,2000-01-03
3,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2000,01,04,2000-01-04
4,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2000,01,05,2000-01-05
...,...,...,...,...,...
2187,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2005,12,27,2005-12-27
2188,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2005,12,28,2005-12-28
2189,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2005,12,29,2005-12-29
2190,/home2/nalex2023/Datasets/ERA_global/era5_pl_q...,2005,12,30,2005-12-30


In [9]:
read_dft_sub = read_dft_sorted[(read_dft_sorted['year'] == '2005')]

In [19]:
read_xr_dset_computed = read_xr_dset['Q'].compute()  

In [24]:
year = 2000
month = 1
#read_xr_dset_computed.to_netcdf(f'/home2/nalex2023/Datasets/tropputs/q_mid_{year}{month:02d}.nc')
read_xr_dset_computed.close()
read_xr_dset.close()


In [ ]:
months = np.arange(1,13,1)
year = 2003


def preprocess_sel_mid(ds):
    ds_sel = ds.sel(level=500)
    return ds_sel


for month in months:
    print(f'Processing year: {year}, month: {month}')
    time_str = f'{year}-{month:02d}'
    subset_dft = read_dft_sorted[(read_dft_sorted['year'] == str(year)) & (read_dft_sorted['month'] == f'{month:02d}')]
    read_xr_dset = xr.open_mfdataset(subset_dft['files'],preprocess=preprocess_sel_mid,chunks={'lat':100,'lon':200,'time':480},
                                 
                                 combine_attrs='drop').sel(time=time_str)
        
        
    read_xr_dset_computed = read_xr_dset['Q'].compute()  

    resampled_dset = read_xr_dset_computed.resample(time='3H').mean()
        
    output_path = f'/home2/nalex2023/Datasets/tropputs/q_mid_{year}{month:02d}.nc'
    resampled_dset.to_netcdf(output_path)
    read_xr_dset.close()
    read_xr_dset_computed.close()
    resampled_dset.close()
    print(f'Saved to {output_path}')
    
        

Processing year: 2003, month: 1


/home1/nalex2023/.local/lib/python3.10/site-packages/xarray/groupers.py:498: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


Saved to /home2/nalex2023/Datasets/tropputs/q_mid_200301.nc
Processing year: 2003, month: 2


/home1/nalex2023/.local/lib/python3.10/site-packages/xarray/groupers.py:498: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


Saved to /home2/nalex2023/Datasets/tropputs/q_mid_200302.nc
Processing year: 2003, month: 3


...

Saved to /home2/nalex2023/Datasets/tropputs/q_mid_200308.nc
Processing year: 2003, month: 9


/home1/nalex2023/.local/lib/python3.10/site-packages/xarray/groupers.py:498: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


Saved to /home2/nalex2023/Datasets/tropputs/q_mid_200309.nc
Processing year: 2003, month: 10


/home1/nalex2023/.local/lib/python3.10/site-packages/xarray/groupers.py:498: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


Saved to /home2/nalex2023/Datasets/tropputs/q_mid_200310.nc
Processing year: 2003, month: 11


In [35]:
read_xr_dset_test = read_xr_dset_computed.resample(freq='3h').mean()

NameError: name 'read_xr_dset_computed' is not defined

In [14]:
read_xr_dset

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 248, latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
    level      float64 8B 500.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time       (time) datetime64[ns] 2kB 2000-01-01 ... 2000-01-31T21:00:00
Data variables:
    Q          (time, latitude, longitude) float32 1GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    utc_date   (time) float64 2kB dask.array<chunksize=(1,), meta=np.ndarray>